# Task 6 — Index and join path

Two questions, both answered from the fact table: which column gets the
index, and what does a snowflake cost against a star for "total weight by
province"? Selectivity = distinct values ÷ rows; a lookup on a value reads
roughly rows-per-value.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows


  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)


registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


In [5]:
# Selectivity for all 8 candidate columns, from the fact table.
cols = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey", "CustomerKey", "ShipmentID"]
sel = pd.DataFrame([{"Column": c,
                     "Distinct": int(fact[c].nunique()),
                     "Selectivity": round(fact[c].nunique() / len(fact), 4),
                     "Rows per value": round(len(fact) / fact[c].nunique(), 1),
                     "Worst value": int(fact[c].value_counts().max())}
                    for c in cols])
print(sel.to_string(index=False))

       Column  Distinct  Selectivity  Rows per value  Worst value
      DateKey        31       0.0258            38.7           40
OriginCityKey         9       0.0075           133.3          605
  DestCityKey        13       0.0108            92.3          175
     RouteKey        21       0.0175            57.1          113
      ModeKey         3       0.0025           400.0          660
    StatusKey         4       0.0033           300.0          821
  CustomerKey        40       0.0333            30.0           57
   ShipmentID      1200       1.0000             1.0            1


In [6]:
# The decision, as numbers.
n = len(fact)
print(f"fact rows: {n}")
print(f"DateKey first: every dashboard page filters on it; {int(fact['DateKey'].nunique())} values; the fact grows by date,")
print("  so new rows append at the end instead of being inserted in the middle — if the table is physically ordered by")
print("  one column, this is the one, because range scans become sequential.")
print(f"CustomerKey second: {int(fact['CustomerKey'].nunique())} values — the best selectivity of any column a human actually filters on.")
print("NOT ShipmentID (perfectly selective, 1.0, but no dashboard query filters on a single consignment — an index nobody")
print("  queries is pure write cost). NOT ModeKey (filtered on every page but 3 values x 400 rows — a lookup still reads")
print("  a third of the table; the engine scans instead).")
print(f"\nwrite cost as a number: {n} additional index-key writes per monthly load for the DateKey index (one per fact row —")
print(f"  write volume on the fact path doubles from {n} to {2 * n} row-writes; ~{n * 16 // 1024} KB index storage at 16 bytes/entry);")
print(f"  a second CustomerKey index adds another {n} writes.")
print("\njoin probes for total weight by province (probes = fact rows x joins traversed):")
print(f"  star: Fact -> DimCity = 2 tables, 1 join, {1 * n:,} probes (DimCity already carries Province/Region denormalised)")
print(f"  snowflake: Fact -> DimCity -> DimProvince -> DimCountry = 4 tables, 3 joins, {3 * n:,} probes (3.0x probes, twice the tables, same answer)")

fact rows: 1200
DateKey first: every dashboard page filters on it; 31 values; the fact grows by date,
  so new rows append at the end instead of being inserted in the middle — if the table is physically ordered by
  one column, this is the one, because range scans become sequential.
CustomerKey second: 40 values — the best selectivity of any column a human actually filters on.
NOT ShipmentID (perfectly selective, 1.0, but no dashboard query filters on a single consignment — an index nobody
  queries is pure write cost). NOT ModeKey (filtered on every page but 3 values x 400 rows — a lookup still reads
  a third of the table; the engine scans instead).

write cost as a number: 1200 additional index-key writes per monthly load for the DateKey index (one per fact row —
  write volume on the fact path doubles from 1200 to 2400 row-writes; ~18 KB index storage at 16 bytes/entry);
  a second CustomerKey index adds another 1200 writes.

join probes for total weight by province (probes = fact 

In [7]:
expected_sel = {"DateKey": (31, 0.0258, 40), "OriginCityKey": (9, 0.0075, 605),
                "DestCityKey": (13, 0.0108, 175), "RouteKey": (21, 0.0175, 113),
                "ModeKey": (3, 0.0025, 660), "StatusKey": (4, 0.0033, 821),
                "CustomerKey": (40, 0.0333, 57), "ShipmentID": (1200, 1.0, 1)}
for _, r in sel.iterrows():
    got = (int(r["Distinct"]), float(r["Selectivity"]), int(r["Worst value"]))
    assert got == expected_sel[r["Column"]], f"{r['Column']}: got {got}, expected {expected_sel[r['Column']]}"
assert 1 * len(fact) == 1200 and 3 * len(fact) == 3600
print("SELF-CHECK PASS — selectivities, worst values, star 1,200 vs snowflake 3,600 probes")

# Analysis
# On 1,200 rows the minutes are almost never an index problem — check first
# (1) a fact table carrying descriptive text instead of keys, (2) a
# relationship set to both directions, (3) a measure recomputing a ratio per
# row; an index rescues none of the three.

SELF-CHECK PASS — selectivities, worst values, star 1,200 vs snowflake 3,600 probes
